# Polars Pipeline — Uçuş Verisi
**Senaryo:** 150.000 satırlık sentetik uçuş verisi üzerinde Polars lazy pipeline  
**Hedef:** Filtre → Group By → Aggregation zinciri + süre/bellek ölçümü

In [1]:
import polars as pl
import numpy as np
import pandas as pd
import time
import os

print(f"polars : {pl.__version__}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")

polars : 1.41.2
pandas : 3.0.3
numpy  : 2.4.6


## 1. Sentetik Veri Üretimi

| Sütun | Tür | Açıklama |
|---|---|---|
| `kalkis_havaalani` | kategorik | IST, SAW, ESB... |
| `varis_havaalani` | kategorik | IST, SAW, ESB... |
| `havayolu` | kategorik | THY, Pegasus, AnadoluJet... |
| `ucus_tipi` | kategorik | ic_hat / dis_hat |
| `kalkis_tarihi` | tarih | 2023-01-01 – 2024-12-31 |
| `bilet_fiyati` | sayısal | Normal dağılım, 300–15000 TL |
| `gecikme_dk` | sayısal | Üstel dağılım, 0–300 dakika |

In [2]:
np.random.seed(42)
N = 150_000

HAVAYOLLARI  = ["THY", "Pegasus", "AnadoluJet", "SunExpress", "Corendon"]
HAVAALANLARI = ["IST", "SAW", "ESB", "ADB", "AYT", "BJV", "TZX", "GZT", "SZF", "VAN"]
UCUS_TIPLERI = ["ic_hat", "dis_hat"]

# Gerçekçi ağırlıklar: IST/SAW en yoğun havalimanı, THY en büyük havayolu
HAY_AGIRLIKLARI = [0.40, 0.30, 0.15, 0.10, 0.05]
HVL_AGIRLIKLARI = [0.35, 0.20, 0.10, 0.10, 0.08, 0.05, 0.04, 0.04, 0.02, 0.02]
TIP_AGIRLIKLARI = [0.65, 0.35]

# 2023-01-01 ile 2024-12-31 arasında rastgele tarih üret
baslangic     = np.datetime64("2023-01-01")
rastgele_gun  = np.random.randint(0, 730, N)
kalkis_tarihi = (baslangic + rastgele_gun.astype("timedelta64[D]")).astype(str)

df_veri = pd.DataFrame({
    "ucus_no":          [f"F{i:06d}" for i in range(1, N + 1)],
    "kalkis_havaalani": np.random.choice(HAVAALANLARI, N, p=HVL_AGIRLIKLARI),
    "varis_havaalani":  np.random.choice(HAVAALANLARI, N),
    "havayolu":         np.random.choice(HAVAYOLLARI, N, p=HAY_AGIRLIKLARI),
    "ucus_tipi":        np.random.choice(UCUS_TIPLERI, N, p=TIP_AGIRLIKLARI),
    "kalkis_tarihi":    kalkis_tarihi,
    "bilet_fiyati":     np.random.normal(2500, 1200, N).clip(300, 15000).round(2),
    "gecikme_dk":       np.random.exponential(20, N).clip(0, 300).round(0).astype(int),
})

os.makedirs("data", exist_ok=True)
df_veri.to_csv("data/ucus_verisi.csv", index=False, encoding="utf-8")

print(f"Satir sayisi  : {len(df_veri):,}")
print(f"Sutun sayisi  : {df_veri.shape[1]}")
print(f"Dosya boyutu  : {os.path.getsize('data/ucus_verisi.csv') / 1024 / 1024:.1f} MB")
df_veri.head(3)

Satir sayisi  : 150,000
Sutun sayisi  : 8
Dosya boyutu  : 7.6 MB


,ucus_no,kalkis_havaalani,varis_havaalani,havayolu,ucus_tipi,kalkis_tarihi,bilet_fiyati,gecikme_dk
0,F000001,IST,IST,AnadoluJet,ic_hat,2023-04-13,1712.75,22
1,F000002,VAN,ADB,SunExpress,ic_hat,2024-03-11,1548.61,1
2,F000003,IST,SAW,Pegasus,ic_hat,2023-09-28,3619.77,34


## 2. Polars Lazy Pipeline — 3 Adımlı Zincir

**Adım 1 — Filtre:** 2024 yılına ait iç hat uçuşları seç  
**Adım 2 — Group By:** Kalkış havalimanı + havayoluna göre grupla  
**Adım 3 — Aggregation:** Toplam gelir, ortalama gecikme, uçuş adedi

In [3]:
t_baslangic = time.perf_counter()

sonuc = (
    pl.scan_csv("data/ucus_verisi.csv")
    # Adim 1: Filtre — 2024 yili + ic hat
    .filter(
        pl.col("kalkis_tarihi").str.starts_with("2024") &
        (pl.col("ucus_tipi") == "ic_hat")
    )
    # Adim 2: Group By — havalimanı + havayolu
    .group_by(["kalkis_havaalani", "havayolu"])
    # Adim 3: Aggregation
    .agg([
        pl.col("bilet_fiyati").sum().alias("toplam_gelir"),
        pl.col("gecikme_dk").mean().alias("ort_gecikme_dk"),
        pl.len().alias("ucus_adedi"),
    ])
    .sort("toplam_gelir", descending=True)
    .collect()
)

t_bitis = time.perf_counter()
sure    = t_bitis - t_baslangic

print(f"Pipeline suresi    : {sure:.3f} saniye")
print(f"Sonuc satir sayisi : {len(sonuc)}")
print(f"Bellek kullanimi   : {sonuc.estimated_size('mb'):.4f} MB")
print()
sonuc

Pipeline suresi    : 0.332 saniye
Sonuc satir sayisi : 50
Bellek kullanimi   : 0.0015 MB



kalkis_havaalani,havayolu,toplam_gelir,ort_gecikme_dk,ucus_adedi
str,str,f64,f64,u32
"""IST""","""THY""",1.6962e7,20.150703,6828
"""IST""","""Pegasus""",1.2640e7,19.314925,5052
"""SAW""","""THY""",9.7716e6,20.297015,3919
"""SAW""","""Pegasus""",7.3892e6,19.594282,2938
"""IST""","""AnadoluJet""",6346649.8,20.150294,2555
…,…,…,…,…
"""VAN""","""SunExpress""",253840.94,17.572917,96
"""GZT""","""Corendon""",245171.16,20.97,100
"""SZF""","""SunExpress""",241997.71,18.631579,95


## 3. Bellek & Süre Özeti

In [4]:
dosya_mb  = os.path.getsize("data/ucus_verisi.csv") / 1024 / 1024
bellek_mb = sonuc.estimated_size("mb")

ozet = pd.DataFrame({
    "Metrik": ["Ham veri (CSV)", "Sonuc DataFrame", "Pipeline suresi"],
    "Deger":  [f"{dosya_mb:.1f} MB", f"{bellek_mb:.4f} MB", f"{sure:.3f} sn"],
})
print(ozet.to_string(index=False))

         Metrik     Deger
 Ham veri (CSV)    7.6 MB
Sonuc DataFrame 0.0015 MB
Pipeline suresi  0.332 sn
